# 02. 数据清洗

**上一步发现的问题**：
1. `Customer ID` 缺失 243,007 条（22.8%）——没有客户 ID 无法做人群分层
2. `Quantity` 有负值——退款单（Invoice 以 C 开头）
3. `Price` 有 0 和负值——异常/调整单
4. 可能存在完全重复的行

**本 notebook 的清洗策略**：
- 剔除 Customer ID 缺失
- 剔除退款单（Quantity < 0 或 Invoice 以 C 开头）——分开统计退款率，但不进入 RFM
- 剔除 Price ≤ 0
- 剔除完全重复行
- 新增 `TotalPrice = Quantity * Price`（订单金额）
- 输出清洗后的 DataFrame 存到 `data/retail_clean.pkl`

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 1. 读入上一步缓存的原始数据

In [2]:
df = pd.read_pickle('../data/retail_raw.pkl')
print('原始形状:', df.shape)
df.head(3)

原始形状: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


## 2. 清洗步骤（分步记录）

每一步都要**记录剔除了多少**，这样最后能产出一张清洗过程表，面试讲故事直接用。

In [3]:
# 用一个 list 记录每一步的保留行数，最后画表
cleaning_log = [('原始数据', len(df))]

# ---- Step 1：剔除 Customer ID 缺失 ----
df = df.dropna(subset=['Customer ID']).copy()
cleaning_log.append(('剔除 Customer ID 缺失后', len(df)))

# ---- Step 2：剔除退款单（Invoice 以 C 开头） ----
mask_refund = df['Invoice'].astype(str).str.startswith('C')
n_refund = mask_refund.sum()
df = df[~mask_refund].copy()
cleaning_log.append((f'剔除退款单（{n_refund} 条）后', len(df)))

# ---- Step 3：剔除 Quantity <= 0（偶发非 C 单的负数/零量） ----
df = df[df['Quantity'] > 0].copy()
cleaning_log.append(('剔除 Quantity ≤ 0 后', len(df)))

# ---- Step 4：剔除 Price <= 0（0 价或调整单） ----
df = df[df['Price'] > 0].copy()
cleaning_log.append(('剔除 Price ≤ 0 后', len(df)))

# ---- Step 5：剔除完全重复行 ----
n_dup = df.duplicated().sum()
df = df.drop_duplicates().copy()
cleaning_log.append((f'剔除重复行（{n_dup} 条）后', len(df)))

# 转成 DataFrame 看一下
log_df = pd.DataFrame(cleaning_log, columns=['步骤', '剩余行数'])
log_df['剔除行数'] = log_df['剩余行数'].shift(1) - log_df['剩余行数']
log_df['保留率'] = (log_df['剩余行数'] / cleaning_log[0][1] * 100).round(2).astype(str) + '%'
log_df

,步骤,剩余行数,剔除行数,保留率
0,原始数据,1067371,NaN,100.0%
1,剔除 Customer ID 缺失后,824364,243007.0,77.23%
2,剔除退款单（18744 条）后,805620,18744.0,75.48%
3,剔除 Quantity ≤ 0 后,805620,0.0,75.48%
4,剔除 Price ≤ 0 后,805549,71.0,75.47%
5,剔除重复行（26124 条）后,779425,26124.0,73.02%


## 3. 新增字段：订单金额 TotalPrice

In [4]:
df['TotalPrice'] = df['Quantity'] * df['Price']

# Customer ID 现在都有值了，转成整数更清爽
df['Customer ID'] = df['Customer ID'].astype(int)

# 为后续分析方便，提取年/月/日/星期
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek  # 0=周一, 6=周日
df['Hour'] = df['InvoiceDate'].dt.hour

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice,Year,Month,YearMonth,DayOfWeek,Hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,2009,12,2009-12,1,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009,12,2009-12,1,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009,12,2009-12,1,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,2009,12,2009-12,1,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,2009,12,2009-12,1,7


## 4. 清洗后再做一次体检

In [5]:
print('清洗后形状:', df.shape)
print('唯一客户数:', df['Customer ID'].nunique())
print('唯一订单数:', df['Invoice'].nunique())
print('唯一商品数:', df['StockCode'].nunique())
print('时间跨度:', df['InvoiceDate'].min(), '→', df['InvoiceDate'].max())
print('\n数值字段体检:')
df[['Quantity', 'Price', 'TotalPrice']].describe()

清洗后形状: (779425, 14)
唯一客户数: 5878
唯一订单数: 36969
唯一商品数: 4631
时间跨度: 2009-12-01 07:45:00 → 2011-12-09 12:50:00

数值字段体检:


,Quantity,Price,TotalPrice
count,779425.000000,779425.000000,779425.000000
mean,13.489370,3.218488,22.291823
std,145.855814,29.676140,227.427075
min,1.000000,0.001000,0.001000
25%,2.000000,1.250000,4.950000
50%,6.000000,1.950000,12.480000
75%,12.000000,3.750000,19.800000
max,80995.000000,10953.500000,168469.600000


In [6]:
# 应该全是 0 了
df.isnull().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
TotalPrice     0
Year           0
Month          0
YearMonth      0
DayOfWeek      0
Hour           0
dtype: int64

## 5. 保存清洗后的数据

后面的 EDA 和 RFM 分析都基于这份干净数据。

In [7]:
df.to_pickle('../data/retail_clean.pkl')
print(f'已保存 ../data/retail_clean.pkl，形状 {df.shape}')

已保存 ../data/retail_clean.pkl，形状 (779425, 14)


---

**本 notebook 完成清洗。核心结论可以直接写进简历/面试材料**：

- 原始数据 106.7 万条，清洗后保留约 XX 万条（保留率 XX%）
- 剔除无客户 ID、退款单、异常价格、重复行等脏数据
- 新增订单金额、时间维度等衍生字段，供后续分析使用

**下一步**：`03_eda.ipynb` 做探索性可视化（按国家、月份、小时、星期拆销售趋势）。